# Liquidity — Risk-Factor Construction Spec (USE4-faithful)

**Purpose.** This notebook is the **source-of-truth spec** for building the
**LIQ** (Liquidity) style factor exactly as MSCI describes it in *Barra USE4
Empirical Notes* (Appendix A, p.52; Section 3.4, p.16). It is **not** a research
notebook. The deliverable is a clean, USE4-faithful LIQ factor written to parquet,
suitable for input to a multi-factor risk model.

**Audience.** You — plus whatever AI assistant you are driving. Treat its output as deeply untrustworthy until audited. Follow the stages linearly. Each stage has:
- ✅ **PDF SPEC** — exact verbatim quote or paraphrase from the USE4 documentation
- ❓ **NOT IN PDF** — something the PDF does not specify; a judgment call you must make
- 💡 **DEFAULT** — a reasonable default for any ❓ item, chosen to match standard Barra practice
- 🧪 **VALIDATE** — sanity checks to run after each stage

**Inputs:** Sharadar SEP daily (volume, sharesbas, close for mcap); estu_monthly for ESTU membership.
No daily price returns, no risk-free rate, no market benchmark — LIQ is constructed
entirely from trading activity data.

**Deliverable:** A parquet file `liq_use4.parquet` keyed by `(permaticker, signal_date)`
containing the standardised LIQ exposure for every stock in the coverage universe on
every monthly signal date.

**Companion specs:** None. LIQ does not depend on other style factors and is not used
as an upstream input by any other factor in the current pipeline.

## §1. The USE4 LIQ spec — verbatim quotes

### 1a. Empirical Notes, Appendix A (p.52) — composite descriptor definition

> **Liquidity (LIQ)**
>
> *Definition:* `0.35 · STOM + 0.35 · STOQ + 0.30 · STOA`
>
> *STOM (Share Turnover, One Month):*
> *"The log of the sum of daily volume divided by shares outstanding over the past month (21 trading days)."*
>
> `STOM_τ = ln( Σ_{t ∈ window_τ} V_t / S_t )`
>
> *STOQ (Share Turnover, One Quarter):*
> *"The log of the average monthly share turnover over the past three months."*
>
> `STOQ = ln( (1/3) · Σ_{τ=1}^{3} exp(STOM_τ) )`
>
> *STOA (Share Turnover, One Year):*
> *"The log of the average monthly share turnover over the past twelve months."*
>
> `STOA = ln( (1/12) · Σ_{τ=1}^{12} exp(STOM_τ) )`
>
> where STOM_τ is the single-month turnover for non-overlapping 21-trading-day window τ
> counted backward from the signal date (τ=1 is the most recent month).

### 1b. Empirical Notes, Section 3.4 (p.16) — economic rationale and weights

> *"Liquidity [captures return] differences due to relative trading activity. ...
> STOM and STOQ receive equal weight (0.35 each) with STOA providing a longer-horizon
> anchor at 0.30."*

### 1c. Methodology Notes, Section 2.3 (p.9) — standardisation rule

> *"Descriptors are standardised to have a mean of 0 and a standard deviation of 1.
> ... μ_l is the cap-weighted mean of the descriptor (within the estimation universe),
> and σ_l is the equal-weighted standard deviation."*

---

**That is all the PDFs say about LIQ construction.**
The PDFs do not specify: handling of zero-volume days, the floor for zero-total-turnover
months, minimum valid monthly windows for STOQ/STOA, the data source for daily shares
outstanding, or outlier treatment for this descriptor specifically.
See §12 for all NOT-IN-PDF decisions.

## §2. End-to-end pipeline

```
┌─────────────────────────────────────────────────────────────────────┐
│  STAGE 0:  Setup, imports, paths                                    │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 1:  Load SEP daily (volume, sharesbas, close)                │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 2:  Build ESTU per signal_date                               │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 4:  LIQ estimator (rolling STOM → STOQ, STOA → composite)   │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 5:  Outlier treatment  (3σ cap-weighted trim)                │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 6:  Standardise  (cap-weighted mean=0, EW std=1)             │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 7:  Save deliverable                                         │
├─────────────────────────────────────────────────────────────────────┤
│  STAGE 8:  Validate                                                 │
└─────────────────────────────────────────────────────────────────────┘
```

**Note:** No Stage 3 (market benchmark) and no RF fetch — LIQ is constructed entirely
from daily trading volume and shares outstanding. No excess returns are involved.

**PIT discipline:** For any signal_date `t`, the only data permitted is data dated ≤ `t`.
All STOM windows look backward from signal_date only.

## §3. Output schema — what the worker delivers

**Raw descriptor column:** `liq`
**Standardised score column:** `liq_score`

**File:** `data/out/liq_use4.parquet`

**Compression:** zstd, statistics=True

**Schema:**

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `signal_date` | Date | End-of-month rebalance date (last trading day of month) |
| `in_estu` | Bool | Whether this stock was in the estimation universe on this date |
| `mcap` | Float64 | Market cap on signal_date (used for cap-weighting) |
| `liq` | Float64 | **Raw descriptor** — composite LIQ = 0.35·STOM + 0.35·STOQ + 0.30·STOA |
| `liq_score` | Float64 | **Final LIQ exposure** — standardised cross-sectionally (the deliverable) |
| `n_obs` | UInt32 | Number of valid monthly STOM windows available (0–12; STOA uses up to 12) |

**Coverage rules:**
- Compute `liq` and `liq_score` for **every stock with sufficient data** (at least STOM
  computable — 1 valid monthly window), not just ESTU members.
- Standardisation moments (mean, std) are computed using **ESTU members only**.
- Non-ESTU stocks get the *same* standardisation parameters applied so the final
  factor is comparable across the coverage universe.

## §4. STAGE 0 — Setup, imports, paths

Standard imports. Use polars, not pandas, throughout (project convention).

✅ **PDF SPEC — LIQ descriptor weights (Empirical Notes, Appendix A, p.52):**

> `W_STOM = 0.35` — weight on STOM (one-month share turnover)\
> `W_STOQ = 0.35` — weight on STOQ (three-month average share turnover)\
> `W_STOA = 0.30` — weight on STOA (twelve-month average share turnover)

✅ **PDF SPEC — window lengths (Empirical Notes, Appendix A, Equations A9–A11):**

> `STOM_WINDOW_TD = 21` — trading days per monthly turnover window\
> `STOQ_MONTHS    = 3`  — number of monthly windows for STOQ\
> `STOA_MONTHS    = 12` — number of monthly windows for STOA

❓ **NOT IN PDF — minimum valid monthly windows for STOQ/STOA:**
The PDF defines STOQ and STOA over exactly 3 and 12 months respectively, but does
not specify what to do if some windows are null (e.g., recently listed stocks).

💡 **DEFAULT:** `MIN_MONTHS_STOQ = 2`, `MIN_MONTHS_STOA = 2` — require at least 2
valid non-null STOM windows for STOQ and STOA; otherwise mark as null. This allows
recently listed stocks to contribute a STOA signal as long as at least 2 months of
data exist.

❓ **NOT IN PDF — minimum observations for final composite:**
The PDF does not set a floor on n_obs for inclusion.

💡 **DEFAULT:** `MIN_OBS = 1` — any stock with at least one valid STOM window (the
most recent month) computes `liq`. If STOQ or STOA cannot be computed, the composite
uses only the available components, renormalised to sum to 1.

```python
# ───── Parameters ─────────────────────────────────────────────────────────────
# ✅ PDF SPEC — descriptor weights (Empirical Notes, Appendix A, p.52)
W_STOM = 0.35     # weight on one-month share turnover
W_STOQ = 0.35     # weight on three-month average share turnover
W_STOA = 0.30     # weight on twelve-month average share turnover

# ✅ PDF SPEC — window lengths (Equations A9-A11)
STOM_WINDOW_TD = 21   # trading days per monthly window
STOQ_MONTHS    = 3    # monthly windows for STOQ
STOA_MONTHS    = 12   # monthly windows for STOA

# 💡 DEFAULT (NOT IN PDF) — minimum valid monthly windows for STOQ/STOA
MIN_MONTHS_STOQ = 2
MIN_MONTHS_STOA = 2

# 💡 DEFAULT (NOT IN PDF) — minimum obs for composite computation
MIN_OBS = 1   # require at least 1 valid STOM window (most recent month)

# 💡 DEFAULT (NOT IN PDF) — calendar start
CALENDAR_START = date(1999, 1, 1)

FACTOR_TYPE = "price_volume"  # uses SEP daily volume+shares, not price returns or SF1
```

## §5. STAGE 1 — Load SEP daily data

**LIQ-specific stage** — loads Sharadar SEP daily data for volume and shares outstanding.
This replaces the standard price-return Stage 1 used by time-series factors.

✅ **PDF SPEC:** The PDF specifies `V_t` (daily trading volume) and `S_t` (shares
outstanding) as the inputs to STOM (Equation A9). No log returns or risk-free rate
are required.

❓ **NOT IN PDF — data source for daily shares outstanding:** The PDF specifies `S_t`
but does not name the data source.

💡 **DEFAULT:** Sharadar SEP `sharesbas` column, which provides basic shares outstanding
at daily frequency. This is the most granular shares series available in the database.

❓ **NOT IN PDF — units consistency:** In Sharadar SEP, `volume` is raw share count and
`sharesbas` is expressed in millions of shares. The daily turnover fraction must use
consistent units.

💡 **DEFAULT:** `turnover_frac_t = volume / (sharesbas * 1_000_000)`. **Verify this
empirically before running at scale** — see Pitfall 3. A quick sanity check: AAPL with
~15 billion shares and ~60 million shares/day volume should give turnover_frac ≈ 0.004.

❓ **NOT IN PDF — zero-volume days:** The PDF sums V_t/S_t over 21 trading days. Days
with zero volume produce a zero contribution to the sum.

💡 **DEFAULT:** Include zero-volume days in the sum. A day with V_t = 0 contributes 0
to Σ(V_t/S_t), correctly reflecting zero trading activity. Do **not** drop or forward-fill
zero-volume rows.

🧪 **VALIDATE:**
- SEP loaded for all permatickers in the coverage universe
- `volume` is non-negative; `sharesbas` is positive for all rows
- Date range extends far enough back from CALENDAR_START to cover 12 × 21 = 252 trading days
  of STOA history (i.e., data starting ≈ 1998)
- Spot check AAPL: daily turnover fraction on a typical day ≈ [0.001, 0.01]
- Spot check a micro-cap: daily turnover fraction on a typical day ≈ [0.0001, 0.005]

## §6. STAGE 2 — Build ESTU per signal_date

**Identical to the shared USE4 build infrastructure** — no factor-specific decisions here.

✅ **PDF SPEC:** Standardisation uses the estimation universe (ESTU) members only for
computing cap-weighted mean and equal-weighted standard deviation.

💡 **DEFAULT:** Load `estu_monthly` (pre-built ESTU membership parquet at
`data/out/estu_use4.parquet`), which provides `(permaticker, signal_date, in_estu, mcap)`
for every monthly signal date.

❓ **NOT IN PDF — ESTU vs coverage universe for LIQ descriptor:** The PDF specifies
standardisation within ESTU but LIQ should be computed for all stocks with sufficient data.

💡 **DEFAULT:** Compute `liq` for all stocks in SEP with at least MIN_OBS valid STOM
windows. Apply ESTU-derived standardisation moments to all stocks including non-ESTU.

🧪 **VALIDATE:**
- ESTU has ~2,500–3,000 members per date in the post-2005 period
- `in_estu` flag is Boolean and non-null for all rows
- `mcap` is positive for all ESTU members

## §7. STAGE 4 — LIQ estimator

✅ **PDF SPEC (Empirical Notes, Appendix A, Equations A9–A11, p.52):**

**STOM** (Share Turnover, One Month — Equation A9):
> *"The log of the sum of daily volume divided by shares outstanding over the past month."*
>
> `STOM_τ = ln( Σ_{t ∈ window_τ} V_t / S_t )`
>
> where window_τ is a non-overlapping 21-trading-day block, τ = 1 being the most recent
> (ending on signal_date), τ = 2 the preceding 21 days, etc.

**STOQ** (Share Turnover, One Quarter — Equation A10):
> *"The log of the average monthly share turnover over the past three months."*
>
> `STOQ = ln( (1/3) · Σ_{τ=1}^{3} exp(STOM_τ) )`
>
> Note: this is the log of the *arithmetic mean* of monthly turnover levels, **not**
> the arithmetic mean of the log-turnover values. Order of operations: compute each
> STOM_τ first, exponentiate, average, then take log.

**STOA** (Share Turnover, One Year — Equation A11):
> *"The log of the average monthly share turnover over the past twelve months."*
>
> `STOA = ln( (1/12) · Σ_{τ=1}^{12} exp(STOM_τ) )`

**Composite:**
> `LIQ = 0.35 · STOM + 0.35 · STOQ + 0.30 · STOA`
>
> where STOM = STOM_1 (the most recent 21-trading-day window).

❓ **NOT IN PDF — handling of zero-total-turnover windows:** If all 21 days in a window
have V_t = 0, the sum is 0 and STOM_τ = ln(0) is undefined.

💡 **DEFAULT:** Mark STOM_τ = null for any window where the total turnover sum is zero
(fully suspended stock). This propagates correctly through STOQ/STOA computation.

❓ **NOT IN PDF — minimum valid windows for partial STOQ/STOA:** A stock with fewer than
3 valid STOM windows cannot compute STOQ as specified; similarly fewer than 12 for STOA.

💡 **DEFAULT:** `MIN_MONTHS_STOQ = 2`, `MIN_MONTHS_STOA = 2`. If a stock has at least 2
valid STOM windows in the STOQ/STOA lookback, compute the average over the available
valid windows (renormalise the denominator accordingly). If fewer than 2 valid windows
exist, mark STOQ or STOA as null.

❓ **NOT IN PDF — composite with partial descriptors:** The formula assumes all three
descriptors are available. If STOQ or STOA is null (newly listed stock), the composite
cannot be computed as specified.

💡 **DEFAULT:** When any sub-descriptor is null, reweight the available descriptors
proportionally to sum to 1.0:
- Only STOM available → `liq = STOM` (effective weight 1.0)
- STOM + STOQ available → `liq = (0.35/0.70)·STOM + (0.35/0.70)·STOQ` (= 0.50, 0.50)
- All three → `liq = 0.35·STOM + 0.35·STOQ + 0.30·STOA`

❓ **NOT IN PDF — trading day calendar:** Window boundaries (multiples of 21 trading days
before signal_date) depend on which days are valid trading days.

💡 **DEFAULT:** Use the calendar of trading dates present in Sharadar SEP itself. A
trading day is any date with at least one row in the SEP daily table. Build the sorted
list once and use `bisect` to find window boundaries.

---
*The section above (PDF SPEC quote through final 💡 DEFAULT) is the `STAGE4_DESCRIPTION`
that `/build-factor` will inject verbatim into the Stage 4 stub docstring. Content below
this line is supplementary guidance for the implementer and is not extracted.*
---

**Implementation notes:**
- Assign each SEP row a window index `w_idx = (td_from_signal) // 21` (integer division of
  trading-day distance from signal_date). All rows with the same `w_idx` belong to the same
  STOM window. Then group by `(permaticker, w_idx)` and sum turnover fractions to get
  Σ(V_t/S_t) per window — fully vectorised, no Python date loop needed.
- For the monthly signal panel, each signal_date needs a backward window of
  12 × 21 = 252 trading days. Pre-filter SEP once (date > signal_date − 252td) then
  assign window indices.
- `n_obs` = number of non-null STOM windows used in the composite (max 12).
- PIT: only SEP rows with `date ≤ signal_date` are permitted.

🧪 **VALIDATE:**
- STOM for a median-liquidity ESTU stock should be approximately [-4, -1]
  (e.g., 0.5%/day turnover × 21 days ≈ 0.105 → STOM ≈ ln(0.105) ≈ -2.25)
- `liq` values across the universe should cluster in [-6, 2]
- High-volume mega-caps (AAPL, NVDA, AMD) should rank in Q5 (most liquid)
- Micro-caps and thinly traded OTC names should rank in Q1
- Spot check on a recent signal_date: median `liq` across ESTU ≈ [-3.5, -2.0]
- MoM Spearman ρ of `liq` between consecutive signal_dates > 0.95

## §8. STAGE 5 — Outlier treatment

❓ **NOT IN PDF for LIQ specifically.** Methodology Notes Section 2.2 (p.8)
describes a general outlier-treatment algorithm that applies to all descriptors:

> *"We trim these observations to three standard deviations from the mean."*

💡 **DEFAULT:** Apply 3σ trimming per signal_date, computed within ESTU using
cap-weighted mean ± 3 × equal-weighted std. Applied to all stocks in the coverage universe.

**Expected clip rate:** LIQ is a log-turnover quantity and approximately log-normal.
Expect ~0.5–2% of ESTU rows clipped per tail.

🧪 **VALIDATE:**
- ~0.5–2% of ESTU rows clipped at each bound per date
- After trimming, post-trim skewness of `liq` within ESTU should be modest (|skew| < 2)
- Trimming should clip in both tails; one-sided clipping suggests a distribution problem

## §9. STAGE 6 — Standardise (cap-weighted mean=0, EW std=1)

✅ **PDF SPEC (Methodology Notes §2.3, p.9):**

> *"μ_l is the cap-weighted mean of the descriptor (within the estimation universe),
> and σ_l is the equal-weighted standard deviation."*

**Per signal_date `t`, using only ESTU members:**

```
μ_t  = Σ_{i ∈ ESTU_t} (mcap_i · liq_i) / Σ mcap_i    # cap-weighted mean
σ_t  = std_{i ∈ ESTU_t}(liq_i)                         # equal-weighted std
liq_score_{i,t} = (liq_i − μ_t) / σ_t                  # applied to ALL i
```

Three critical points:
1. Moments estimated on **ESTU only**; applied to **every stock** in the coverage universe.
2. Mean is **cap-weighted**; std is **equal-weighted**.
3. Cap-weighting the mean ensures the cap-weighted market portfolio has zero LIQ exposure.

🧪 **VALIDATE:**
- `Σ_{i ∈ ESTU} (mcap_i · liq_score_i) / Σ mcap_i ≈ 0` (cap-weighted mean = 0 by construction)
- `std_{i ∈ ESTU}(liq_score_i) ≈ 1` (equal-weighted std = 1 by construction)
- Fraction of `liq_score` values in [−3, +3] > 95%

## §10. STAGE 7 — Save deliverable

Write the final panel to `data/out/liq_use4.parquet`.
Compression: zstd. Statistics: True.

Include `liq` (raw composite) for downstream audit and model calibration.
The downstream consumer will use `liq_score`.

## §11. STAGE 8 — Validation

### Required checks (all must pass before signing off)

**1. Standardisation moments on ESTU:**
```
cap-weighted mean of liq_score ≈ 0   (|mean| < 1e-6)
equal-weighted std of liq_score ≈ 1   (|std - 1| < 0.02)
```

**2. Raw descriptor sanity:**
```
liq values in [-6, 2] for ≥ 95% of ESTU rows
(values outside this range signal unit errors in V_t/S_t computation)
```

**3. Coverage stability:**
```
≥ 4,000 stocks with non-null liq_score per date post-2005
```

**4. Factor stability (month-over-month rank correlation):**
```
MoM Spearman ρ > 0.95
(liquidity is highly persistent — high ρ is expected and required)
```

**5. Economic direction:**
```
High-volume mega-caps (AAPL, NVDA) should appear in top quintile (Q5, liq_score > 1)
Micro-caps and OTC names with sparse trading should appear in bottom quintile (Q1, liq_score < −1)
```

**6. Disk vs memory equivalence:**
```
max abs diff of liq values between written and read-back file < 1e-10
```

---

**Shared validation conventions (all factors, 2026-06-10):**
- **Coverage (check 3)** is evaluated on **completed months only** — the final
  signal date is the in-progress month (freshest data can lag the Sharadar
  refresh) and is excluded from the floor.
- **Fraction of scores in [−3, +3]** is reported for the full universe *and* for
  ESTU; the ESTU figure is the operationally relevant one (non-ESTU micro-caps
  pull the all-universe tail).
- Common check machinery lives in `common/ (your shared utilities)`
  .


## §12. Master list of ❓ NOT-IN-PDF decisions

| # | Decision | Default chosen | Alternative | Where to revisit |
|---|---|---|---|---|
| 1 | Zero-volume days (V_t = 0) | Include in sum as V_t/S_t = 0 | Drop and exclude from window | Stage 4 — zero trading is genuine illiquidity; including it is economically correct |
| 2 | Fully-suspended month (all-zero volume) | STOM_τ = null | Apply ln(epsilon) floor | Stage 4 — null propagates cleanly; arbitrary floor creates scale distortion |
| 3 | Minimum valid windows for STOQ | 2 of 3 | 3 of 3 (strict PDF) or 1 of 3 | Stage 4 — 2/3 allows recent IPOs to contribute a STOQ signal |
| 4 | Minimum valid windows for STOA | 2 of 12 | 12 of 12 (strict) or 1 of 12 | Stage 4 — 2/12 is permissive; tradeoff between coverage and signal quality |
| 5 | Composite with partial descriptors | Renormalise available weights to 1.0 | Mark composite null if any sub-descriptor missing | Stage 4 — renormalisation preserves coverage for newer stocks |
| 6 | Daily shares outstanding source | Sharadar SEP `sharesbas` | Static quarterly from SF1 | Stage 1 — SEP daily is more granular; captures intra-quarter share changes |
| 7 | Units for V_t/S_t computation | `volume / (sharesbas * 1e6)` | Verify actual Sharadar units | Stage 1 — **must verify empirically** before running; see Pitfall 3 |
| 8 | Trading day calendar | Dates present in Sharadar SEP | NYSE calendar from external library | Stage 1 — SEP-native calendar is automatically PIT-consistent and avoids extra dependency |
| 9 | Outlier treatment | 3σ cap-weighted trim (USE4 standard) | Different clip bounds | Stage 5 — standard USE4 default applied to all descriptors |
| 10 | Minimum obs for composite inclusion | MIN_OBS = 1 (any valid STOM window) | MIN_OBS = 3 or more | Stage 4 — single STOM allows maximum coverage; raise if signal quality concerns emerge |

## §13. Common pitfalls — read this before coding

**Pitfall 1: Log of mean vs mean of log (the most critical mistake)**
STOQ and STOA compute `ln(mean(exp(STOM_τ)))` — the log of the *arithmetic mean* of
turnover *levels*, not the arithmetic mean of log-turnover values. Computing
`mean(STOM_τ)` directly (averaging the logs) produces the geometric mean of turnover,
which is systematically lower and produces a different factor entirely. This mistake is
not obvious from a cursory reading of Equations A10 and A11.

**Pitfall 2: Zero-volume day dropping**
Zero-volume days (V_t = 0) contribute zero to Σ(V_t/S_t) and must be included in the sum.
Dropping them as null/missing shrinks the effective window and inflates STOM artificially.
A suspension day with no trading should reduce monthly turnover, not be ignored.

**Pitfall 3: Sharesbas unit mismatch (highest risk)**
In Sharadar SEP, `sharesbas` is denominated in millions of shares while `volume` is in
raw share count. The ratio `volume / sharesbas` is off by a factor of 1,000,000, producing
STOM values ≈ +11 instead of ≈ −2. Always compute `volume / (sharesbas * 1_000_000)`.
Empirical check: AAPL with ~15 billion shares and ~60 million shares of daily volume should
yield turnover_frac ≈ 0.004 and STOM ≈ ln(0.004 × 21) ≈ ln(0.084) ≈ −2.5. If you see
STOM > 0 for typical stocks, the units are wrong.

**Pitfall 4: STOM window boundary alignment**
Window τ=1 ends on signal_date (inclusive) and covers the 21 most recent trading days.
Window τ=2 covers the 21 trading days immediately before window τ=1. Windows must be
non-overlapping and contiguous. Using calendar-month boundaries (Jan 1 – Jan 31) instead
of fixed 21-trading-day blocks gives inconsistent window sizes and breaks the PIT guarantee
at signal dates in the middle of a calendar month.

**Pitfall 5: PIT with signal_date at month-end**
The STOM_1 window [signal_date − 20td, signal_date] is fully PIT-compliant when computed
correctly. Sloppy date arithmetic (e.g., `date < signal_date + timedelta(days=1)` with
calendar days vs trading days) can accidentally include the next trading day's data. Always
filter SEP by trading-day index, not calendar date arithmetic.

**Pitfall 6: Renormalised composites for new stocks are cross-sectionally different**
A stock with only STOM available gets `liq = STOM` (weight 1.0), while a full-window
stock uses the 0.35/0.35/0.30 blend. These are measuring slightly different things.
The `n_obs` column captures how many months of history were available — use it to flag
or exclude thin-history rows in downstream analysis if needed.

## §14. Final summary — what "done" looks like

**The deliverable is one file:** `data/out/liq_use4.parquet`

**Acceptance criteria:**

1. ✅ Schema matches §3 (7 columns, correct types)
2. ✅ All 6 required validation checks in §11 pass within tolerance
3. ✅ ESTU has ~2,500–3,000 names per date, stable over time
4. ✅ Cap-weighted mean of `liq_score` is machine-zero for every date in ESTU
5. ✅ Equal-weighted std of `liq_score` is 1.0 (to 2 decimal places) for every date in ESTU
6. ✅ ≥ 95% of ESTU `liq` values fall in [−6, 2]
7. ✅ Coverage of `liq_score` ≥ 4,000 stocks per date post-2005
8. ✅ Month-over-month rank stability ρ > 0.95
9. ✅ All ❓ NOT-IN-PDF decisions in §12 are documented somewhere in the code

**Once LIQ is built and passes validation, the next steps are:**
LIQ does not feed as an upstream input into any other style factor in the current
pipeline. It can be directly consumed by the multi-factor risk model alongside Beta,
Size, Value (YILD), Momentum, Leverage, Growth, and other completed style factors.